# Wave 2A controlled single-factor experiments on Colab

Run the seven Wave 2A controlled arms on the promoted `arm4_e2e_unseen_attn_aux / w2c_exp1_clean_sep_w10` base (single-stage E2E `Wave2BE2ERanker`, `cand_42 + CUST_7 + SEQ_5`, Pure-Q `Q = v_bn`, Full-42 cross-attention bias, clean conflict-free separated Main/Option A targets, `OptionAUnseenHead` on `cand_37` with **0% dropout**, **`1.0 * L_optA`**, per-epoch OOF affine blend selection):

- **Promoted Reference (`v2_full_reference.json`):** `w2c_exp1_clean_sep_w10` (`cal-Discovery == cal-Unseen` Ep 6: `R@10 = 82.12%`, `R@25 = 92.77%`, `NDCG@10 = 0.65380`, `New@10 = 1,219`, `PureU@10 = 2,298`, `PureU@25 = 3,218`).
1. `w2_2mem_blocks` — 2× stacked `[Self-Attention → FFN(384)]` memory blocks (`d=128`, `(384, 192, 96)` MLPs).
2. `w2_wider_mlp` — Wider MLPs (`memory_layers=1`, `memory_ffn_dim=512`, `(512, 256, 128)` scoring MLPs, `d=128`).
3. `w2_2mem_wider_mlp` — Both 2× `[Self-Attention → FFN(512)]` AND wider `(512, 256, 128)` scoring MLPs (`d=128`).
4. `w2_dim192` — Bigger product & customer representations (`d=192`, `num_heads=8` [24D/head], `memory_ffn_dim=576`, `memory_layers=1`, `(384, 192, 96)` MLPs).
5. `w2_hist_len64` — Longer 64-event purchase history (`seq_len=64`).
6. `w2_recency_hl365` — Exponential recency sample weighting (`half_life=365d`, `min_weight=0.1`).
7. `w2_2mem_wider_mlp_syn7030` — Separate bigger-model experiment: `w2_2mem_wider_mlp` + exact 70/30 masked-`b*` & FO synthetic unseen augmentation (`synthetic_unseen.py`, `w_optA=1.0`, `skip_main_head=True`).

Training ends on January 31, 2026 (`< 2026-02-01`). Validation runs from February 1 through June 30, 2026 (`16,623` baskets, zero held-out Test `>= 2026-07-01` access).

Select a **v6e-1 TPU** (recommended — 918 TFLOPS native `bf16`, 32 GB HBM, ~173 GB RAM) or a native-`bf16` CUDA GPU (**L4 / G4 / A100**) with high system RAM. Place the following files in `MyDrive/v2_full_colab/`:

- `v2_full_colab_bundle.tar.zst` and `v2_full_reviewed_code.tar.gz` from `exports/`.
- `commercial_dev_events.parquet` from `data/v2_full/interim/` and `customers.csv` from `data/v2_full/raw/`.

Completed arms and comparison reports are synced automatically to `outputs/`.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import shutil
import psutil
import torch
from google.colab import drive

ACCEL_DEVICE = None
ACCEL_DESC = None
if torch.cuda.is_available():
    if not torch.cuda.is_bf16_supported(including_emulation=False):
        raise RuntimeError(f"GPU {torch.cuda.get_device_name(0)} lacks native hardware bf16. Select v6e-1 TPU, L4, G4, or A100.")
    ACCEL_DEVICE = "cuda"
    ACCEL_DESC = f"CUDA GPU: {torch.cuda.get_device_name(0)}"
else:
    # Probe TPU in a short-lived subprocess so the parent notebook never holds an exclusive PJRT/libtpu lock.
    probe_code = (
        "import os, torch; "
        "os.environ.setdefault('PJRT_DEVICE', 'TPU'); "
        "dev = None; kind = None\n"
        "try:\n"
        "    import torch_tpu; dev = torch.device('tpu'); kind = 'tpu'\n"
        "except Exception:\n"
        "    import torch_xla\n"
        "    dev = torch_xla.device() if hasattr(torch_xla, 'device') else __import__('torch_xla.core.xla_model', fromlist=['xla_device']).xla_device()\n"
        "    kind = 'xla'\n"
        "x = torch.ones(2, dtype=torch.bfloat16, device=dev); print(f'{kind}|{dev}|{float((x + x).sum().cpu())}')"
    )
    probe = subprocess.run([sys.executable, "-c", probe_code], capture_output=True, text=True)
    if probe.returncode != 0:
        raise RuntimeError(
            "No native-bf16 GPU (L4/G4/A100) or active TPU (v6e-1/v5e-1) found. "
            f"TPU probe stderr: {probe.stderr.strip()[-400:]}"
        )
    last_line = [ln for ln in probe.stdout.splitlines() if "|" in ln][-1]
    ACCEL_DEVICE, dev_str, _ = last_line.split("|", 2)
    ACCEL_DESC = f"TPU ({ACCEL_DEVICE}): {dev_str}"

print(f"Verified native-bf16 accelerator -> {ACCEL_DESC}", flush=True)
drive.mount("/content/drive")
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True)

In [ ]:
import hashlib

drive_dir = Path("/content/drive/MyDrive/v2_full_colab")
repo_dir = Path("/content/repo")
sync_dir = drive_dir / "outputs"
patch = drive_dir / "v2_full_reviewed_code.tar.gz"
expected_sha = "b4e521f566d898b4468bf85712d7fa5a0b221289952f5ffc740ddf1d91c1c070"
actual_sha = hashlib.sha256(patch.read_bytes()).hexdigest()
if actual_sha != expected_sha:
    raise ValueError(
        f"Upload the reviewed Wave 2A code archive ({expected_sha[:12]}); found {actual_sha[:12]}."
    )

inputs = {"interim": "commercial_dev_events.parquet", "raw": "customers.csv"}
for name in inputs.values():
    if not (drive_dir / name).is_file():
        raise FileNotFoundError(drive_dir / name)
repo_dir.mkdir(parents=True, exist_ok=True)
os.chdir(repo_dir)
if not Path("data/v2_full/interim/prepared_v2_full/manifest.json").is_file():
    bundle = drive_dir / "v2_full_colab_bundle.tar.zst"
    bundle_expected_sha = "a6201348dce01ae483dde470c5a279806f639e385b54f85d21977f44cc384c06"
    h = hashlib.sha256()
    with bundle.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    if h.hexdigest() != bundle_expected_sha:
        raise ValueError(f"Unexpected bundle SHA-256: {h.hexdigest()[:12]} != {bundle_expected_sha[:12]}")
    subprocess.run(["tar", "--zstd", "-xf", bundle], check=True)
for sub in ("checkpoints/v2_full", "experiments/v2_full"):
    if (sync_dir / sub).is_dir():
        shutil.copytree(sync_dir / sub, sub, dirs_exist_ok=True)
subprocess.run(["tar", "-xzf", patch], check=True)
for folder, name in inputs.items():
    shutil.copy2(drive_dir / name, Path("data/v2_full") / folder / name)
print("Workspace ready with Wave 2A patch:", actual_sha[:12], flush=True)


In [ ]:
def run_experiment(module, *args):
    cmd = [sys.executable, "-m", f"recommender.experiments.v2_full.{module}", *args]
    print(">>", " ".join(cmd), flush=True)
    env = {**os.environ, "PYTHONPATH": ".", "PYTHONUNBUFFERED": "1"}
    if ACCEL_DEVICE == "xla":
        env.setdefault("PJRT_DEVICE", "TPU")
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        rc = proc.wait()
    except BaseException:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
        raise
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

avail_gb = psutil.virtual_memory().available / (1024**3)
print(f"Available RAM: {avail_gb:.1f} GB | Accelerator: {ACCEL_DESC}", flush=True)
if avail_gb < 32.0:
    raise RuntimeError(f"High-RAM runtime required (>= 32 GB free for --in-memory global shuffle); found {avail_gb:.1f} GB.")
if ACCEL_DEVICE in ("xla", "tpu"):
    print("Note: first 1-2 TPU steps compile the static XLA graph (~30-60s), then epochs run at full v6e-1 speed.", flush=True)
common = ["--device", ACCEL_DEVICE, "--seed", "42", "--skip-v1-verify", "--in-memory", "--sync-dir", str(sync_dir)]
run_experiment("run_v2_full_wave2", "--amp", *common)

In [ ]:
import json
import pandas as pd

evidence = Path("experiments/v2_full")
rows, intervals = [], []


def add_row(model, evaluation, pure):
    metrics = evaluation["bn_metrics"]
    sku = evaluation["resolved_sku_metrics"]
    unseen = pure["overall"]
    rows.append({
        "Model": model,
        "Ep": evaluation.get("epoch"),
        "Recall@5": metrics["bn_recall@5"],
        "Recall@10": metrics["bn_recall@10"],
        "NDCG@10": metrics["bn_ndcg@10"],
        "New hits@10": metrics["true_new_bn_hits@10"],
        "FO new hits@10": evaluation["cohorts"]["first_observed"]["true_new_bn_hits@10"],
        "Ret new hits@10": evaluation["cohorts"]["returning"]["true_new_bn_hits@10"],
        "Pure U@10": unseen["pure_unseen_hits@10"],
        "Pure U@25": unseen["pure_unseen_hits@25"],
        "PU NDCG@10": unseen["pure_unseen_ndcg@10_new_baskets"],
        "SKU recall@10": sku["sku_recall@10"],
        "SKU NDCG@10": sku["sku_ndcg@10"],
        "SKU new@10": sku["sku_new_hits@10"],
    })


def add_intervals(source, comparisons):
    for name, comparison in comparisons.items():
        if comparison is None:
            continue
        for metric, delta in comparison.items():
            low, _, high = delta["ci_95"]
            intervals.append({
                "Source": source,
                "Comparison": name,
                "Metric": metric,
                "Difference": delta["point_delta"],
                "95% low": low,
                "95% high": high,
            })


ref_arm = Path("checkpoints/v2_full/wave2c_arm4_e2e_unseen_attn_aux-seed42/w2c_exp1_clean_sep_w10/arm_result.json")
if ref_arm.is_file():
    ref_doc = json.loads(ref_arm.read_text())
    add_row("REF: w2c_exp1_clean_sep_w10 (cal-Disc==Unseen Ep 6)", ref_doc["optionA_oof_discovery"], ref_doc["optionA_oof_discovery"]["pure_unseen_head"])
    add_row("REF: w2c_exp1_clean_sep_w10 (cal-NDCG Ep 10)", ref_doc["optionA_oof_ndcg"], ref_doc["optionA_oof_ndcg"]["pure_unseen_head"])

base = "arm4_e2e_unseen_attn_aux"
result = json.loads((evidence / f"v2_full_wave2a_{base}_clean_sep_w10_seed42.json").read_text())
for name, arm in result["arms"].items():
    for key, label in (
        ("optionA_oof_discovery", "cal-Discovery"),
        ("optionA_oof_ndcg", "cal-NDCG"),
        ("optionA_oof_unseen", "cal-Unseen"),
    ):
        if key in arm:
            ev = arm[key]
            add_row(f"{name} ({label})", ev, ev["pure_unseen_head"])
add_intervals("Wave 2A", result["paired_bootstrap_95ci"])
if result.get("skipped"):
    print("Skipped arms:", result["skipped"])

display(pd.DataFrame(rows))
display(pd.DataFrame(intervals))
